### **Semana 7 - Evaluación de retrieval y RAG**

#### **Pregunta experimental**

> ¿Cómo cambia la calidad del ranking, el comportamiento de la generación y el costo del sistema cuando se incorporan recuperación léxica y reranking al pipeline denso, manteniendo fijo el benchmark?

```text
A = dense

B = dense + BM25 + RRF

C = dense + BM25 + RRF + Cross-Encoder
```

Esta semana cambia el criterio de éxito:

```text
"se ve bien" != evidencia experimental
```

El cuaderno separa:

```text
retrieval   -> Recall@k/MRR/nDCG
generation  -> correctness/citations/faithfulness
system      -> latency/failure modes
```


#### **0. Protocolo antes de ejecutar**

Completa antes de observar los resultados:

```text
Hipótesis H1:
La fusión híbrida recuperará evidencia que dense puede perder.

Hipótesis H2:
El reranker modificará principalmente el orden de candidatos ya recuperados.

Hipótesis H3:
Una mejora de retrieval no garantizará una mejora equivalente de generation.

Métrica principal de ranking:
MRR + nDCG@5

Métrica de cobertura:
Recall@3/Recall@5

Incertidumbre:
bootstrap pareado sobre diferencias por consulta
IC 95%, 5000 remuestreos

Costo:
latency_total_mean_ms/latency_total_p95_ms +
desglose por dense/BM25/RRF/reranker

Limitación conocida:
24 consultas docentes no constituyen un benchmark de producción.
Un IC bootstrap cuantifica incertidumbre condicionada a estas consultas,
no convierte el benchmark pequeño en evidencia universal.
```

No se modifican consultas, qrels ni respuestas de referencia después de observar qué condición gana.


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import re
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

SEED = 42
np.random.seed(SEED)

TARGET_WORDS = 180
OVERLAP_PASSAGES = 1
CANDIDATE_DEPTH = 10
TOP_K_VALUES = [1, 3, 5]
RRF_CONSTANT = 60
RAG_TOP_K = 3  # Contexto canónico: alineado con Recall@3 y fijo en A/B/C.
RAG_CONTEXT_TOP_K_VALUES = [1, 3, 5]
BOOTSTRAP_RESAMPLES = 5000
BOOTSTRAP_ALPHA = 0.05

DENSE_MODEL_ID = "intfloat/multilingual-e5-small"
RERANKER_MODEL_ID = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
GENERATOR_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

RUN_REAL_RETRIEVAL = os.getenv("CC0F4_RUN_REAL_RETRIEVAL", "1") == "1"
RUN_REAL_RERANKER = os.getenv("CC0F4_RUN_REAL_RERANKER", "1") == "1"
RUN_REAL_LLM = os.getenv("CC0F4_RUN_REAL_LLM", "0") == "1"

AUDIT_QUERY_IDS = ["Q04", "Q05", "Q10", "Q15", "Q21", "Q22"]

print("Python:", sys.version.split()[0])
print("RUN_REAL_RETRIEVAL:", RUN_REAL_RETRIEVAL)
print("RUN_REAL_RERANKER:", RUN_REAL_RERANKER)
print("RUN_REAL_LLM:", RUN_REAL_LLM)


#### **1. Cargar el benchmark congelado**

Semana 7 reutiliza exactamente los datos de Semana 4 y añade respuestas de referencia.

```text
Semana4/datos/
  corpus_semana4.jsonl
  queries_semana4.jsonl
  qrels_semana4.json

Semana7/datos/
  reference_answers_semana7.jsonl
```

Los qrels continúan definidos sobre `passage_id`.


In [ ]:
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"JSON inválido en {path}, línea {line_number}"
                ) from exc
    return rows


def resolve_paths() -> tuple[Path, Path, Path]:
    cwd = Path.cwd().resolve()

    explicit4 = os.getenv("CC0F4_SEMANA4_DATA")
    explicit7 = os.getenv("CC0F4_SEMANA7_DATA")
    explicit_results = os.getenv("CC0F4_SEMANA7_RESULTS")

    data4 = Path(explicit4).expanduser().resolve() if explicit4 else None
    data7 = Path(explicit7).expanduser().resolve() if explicit7 else None
    results = Path(explicit_results).expanduser().resolve() if explicit_results else None

    roots = [cwd, *cwd.parents]

    if data4 is None:
        for root in roots:
            candidate = root / "Semana4" / "datos"
            if (candidate / "corpus_semana4.jsonl").is_file():
                data4 = candidate
                break

    if data7 is None:
        for root in roots:
            candidate = root / "Semana7" / "datos"
            if (candidate / "reference_answers_semana7.jsonl").is_file():
                data7 = candidate
                break
        if data7 is None and (cwd / "datos" / "reference_answers_semana7.jsonl").is_file():
            data7 = cwd / "datos"

    if results is None:
        for root in roots:
            candidate = root / "Semana7"
            if candidate.is_dir():
                results = candidate / "resultados"
                break
        if results is None:
            results = cwd / "resultados"

    if data4 is None or data7 is None:
        raise FileNotFoundError(
            "No se encontraron los datos. Ejecuta desde la raíz de CC-0F4 "
            "o define CC0F4_SEMANA4_DATA y CC0F4_SEMANA7_DATA."
        )

    results.mkdir(parents=True, exist_ok=True)
    return data4, data7, results


DATA4_DIR, DATA7_DIR, RESULTS_DIR = resolve_paths()

documents = read_jsonl(DATA4_DIR / "corpus_semana4.jsonl")
queries = read_jsonl(DATA4_DIR / "queries_semana4.jsonl")
qrels = json.loads((DATA4_DIR / "qrels_semana4.json").read_text(encoding="utf-8"))
references = read_jsonl(DATA7_DIR / "reference_answers_semana7.jsonl")

query_by_id = {row["query_id"]: row for row in queries}
reference_by_id = {row["query_id"]: row for row in references}

print("DATA4_DIR:", DATA4_DIR)
print("DATA7_DIR:", DATA7_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("Documentos:", len(documents))
print("Consultas:", len(queries))
print("Qrels:", len(qrels))
print("Referencias:", len(references))


In [ ]:
def validate_benchmark() -> None:
    query_ids = [row["query_id"] for row in queries]
    reference_ids = [row["query_id"] for row in references]

    if len(query_ids) != len(set(query_ids)):
        raise ValueError("query_id duplicados")
    if set(query_ids) != set(qrels):
        raise ValueError("queries y qrels no tienen los mismos IDs")
    if set(query_ids) != set(reference_ids):
        raise ValueError("queries y reference_answers no tienen los mismos IDs")

    passage_ids = {
        passage["passage_id"]
        for doc in documents
        for passage in doc["passages"]
    }

    for qid, relevant in qrels.items():
        if not relevant:
            raise ValueError(f"qrels vacío para {qid}")
        missing = set(relevant) - passage_ids
        if missing:
            raise ValueError(f"{qid}: passages inexistentes en qrels: {sorted(missing)}")

    for row in references:
        qid = row["query_id"]
        supporting = set(row["supporting_passage_ids"])
        if not supporting <= set(qrels[qid]):
            raise ValueError(
                f"{qid}: supporting_passage_ids debe ser subconjunto de qrels"
            )

    print("Benchmark: OK")


validate_benchmark()


#### **2. Chunking y unidad de evaluación**

El sistema recupera chunks, pero la relevancia está definida sobre passages.

Por ello cada chunk conserva:

```text
chunk_id
passage_ids
text
```

y antes de calcular las métricas expandimos el ranking de chunks a un ranking de `passage_id` sin duplicados.

Esto evita contar dos veces una evidencia repetida por overlap.


In [ ]:
def count_words(text: str) -> int:
    return len(re.findall(r"\S+", text))


def build_chunks(
    documents: list[dict[str, Any]],
    target_words: int = TARGET_WORDS,
    overlap_passages: int = OVERLAP_PASSAGES,
) -> list[dict[str, Any]]:
    chunks = []

    for doc in documents:
        passages = doc["passages"]
        start = 0
        chunk_number = 1

        while start < len(passages):
            selected = []
            words = 0
            i = start

            while i < len(passages) and (words < target_words or not selected):
                selected.append(passages[i])
                words += count_words(passages[i]["text"])
                i += 1

            chunks.append({
                "chunk_id": f'{doc["doc_id"]}-C{chunk_number:02d}',
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "passage_ids": [p["passage_id"] for p in selected],
                "text": "\n".join(p["text"] for p in selected),
            })

            if i >= len(passages):
                break

            next_start = max(start + 1, i - overlap_passages)
            start = next_start
            chunk_number += 1

    return chunks


chunks = build_chunks(documents)
chunk_by_id = {chunk["chunk_id"]: chunk for chunk in chunks}

print("Chunks:", len(chunks))
pd.DataFrame([
    {
        "chunk_id": c["chunk_id"],
        "passage_ids": ", ".join(c["passage_ids"]),
        "words": count_words(c["text"]),
    }
    for c in chunks
]).head()


In [ ]:
def expand_to_passage_ranking(
    ranked_chunks: list[dict[str, Any]],
) -> list[str]:
    ordered = []
    seen = set()

    for chunk in ranked_chunks:
        for passage_id in chunk["passage_ids"]:
            if passage_id not in seen:
                ordered.append(passage_id)
                seen.add(passage_id)

    return ordered


# Demostración del problema de overlap.
demo = [
    {"passage_ids": ["P1", "P2"]},
    {"passage_ids": ["P2", "P3"]},
]
assert expand_to_passage_ranking(demo) == ["P1", "P2", "P3"]
print("Deduplicación por passage_id: OK")


#### **3. Métricas desde cero**

Las implementaciones se mantienen pequeñas para que las definiciones sean visibles.

$$
Recall@k(q) =
\frac{|R_q \cap L_q^{(k)}|}
{|R_q|}
$$

$$
RR(q)=\frac{1}{rank\ de\ la\ primera\ evidencia\ relevante}
$$

$$
nDCG@k=\frac{DCG@k}{IDCG@k}
$$

El experimento utiliza relevancia binaria porque los qrels existentes son binarios.


In [ ]:
def recall_at_k(
    passage_ranking: list[str],
    relevant_passages: list[str],
    k: int,
) -> float:
    relevant = set(relevant_passages)
    retrieved = set(passage_ranking[:k])
    return len(relevant & retrieved) / len(relevant)


def reciprocal_rank(
    passage_ranking: list[str],
    relevant_passages: list[str],
) -> float:
    relevant = set(relevant_passages)
    for rank, passage_id in enumerate(passage_ranking, start=1):
        if passage_id in relevant:
            return 1.0 / rank
    return 0.0


def dcg_at_k(
    passage_ranking: list[str],
    relevant_passages: list[str],
    k: int,
) -> float:
    relevant = set(relevant_passages)
    total = 0.0
    for i, passage_id in enumerate(passage_ranking[:k], start=1):
        rel = 1.0 if passage_id in relevant else 0.0
        gain = (2.0 ** rel) - 1.0
        total += gain / math.log2(i + 1)
    return total


def ndcg_at_k(
    passage_ranking: list[str],
    relevant_passages: list[str],
    k: int,
) -> float:
    dcg = dcg_at_k(passage_ranking, relevant_passages, k)
    ideal_ranking = list(relevant_passages)
    idcg = dcg_at_k(ideal_ranking, relevant_passages, k)
    return dcg / idcg if idcg > 0 else 0.0


# Tests pequeños: misma cobertura, distinto orden.
ranking_a = ["R", "X", "Y", "Z"]
ranking_b = ["X", "Y", "R", "Z"]
relevant = ["R"]

assert recall_at_k(ranking_a, relevant, 4) == recall_at_k(ranking_b, relevant, 4) == 1.0
assert reciprocal_rank(ranking_a, relevant) > reciprocal_rank(ranking_b, relevant)
assert ndcg_at_k(ranking_a, relevant, 4) > ndcg_at_k(ranking_b, relevant, 4)

print("Tests de métricas: OK")


#### **Actividad 1 - Predice antes de ejecutar**

Dos sistemas recuperan la misma evidencia dentro de `top-5`.

```text
A: relevante en rank 1
B: relevante en rank 4
```

Antes de ejecutar responde:

1. ¿Recall@5 cambia?
2. ¿MRR cambia?
3. ¿nDCG@5 cambia?
4. ¿qué métrica describe mejor el problema de ranking?.


#### **4. Recuperación léxica y dense retrieval**

Semana 7 conserva los componentes de Semana 5.

El modo real utiliza E5 y Cross-Encoder.

El modo offline usa sustitutos ligeros únicamente para validar el pipeline:

```text
offline dense
-> TF-IDF + SVD

offline reranker
-> overlap léxico
```

Por tanto:

```text
offline metrics != evidencia de E5/Cross-Encoder
```


In [ ]:
TOKEN_RE = re.compile(r"[a-záéíóúñü0-9]+", re.IGNORECASE)

BM25_PRESERVED_TERMS = frozenset({
    "no", "sin", "si", "puede", "pueden", "debe", "más",
})

SPANISH_STOPWORDS = frozenset({
    "a", "al", "algo", "como", "con", "cuando", "de", "del", "donde",
    "e", "el", "ella", "ellas", "ellos", "en", "entre", "era", "es",
    "esa", "ese", "eso", "esta", "este", "esto", "ha", "hay", "la",
    "las", "le", "les", "lo", "los", "o", "para", "pero", "por",
    "que", "qué", "se", "su", "sus", "también", "un", "una", "uno",
    "unos", "unas", "y", "ya",
})


def tokenize_bm25(text: str) -> list[str]:
    tokens = [m.group(0).lower() for m in TOKEN_RE.finditer(text)]
    return [
        token
        for token in tokens
        if token in BM25_PRESERVED_TERMS or token not in SPANISH_STOPWORDS
    ]


class SimpleBM25:
    def __init__(self, texts: list[str], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.docs = [tokenize_bm25(text) for text in texts]
        self.lengths = np.array([len(doc) for doc in self.docs], dtype=float)
        self.avgdl = float(self.lengths.mean()) if len(self.lengths) else 1.0
        self.tf = [Counter(doc) for doc in self.docs]

        df = Counter()
        for doc in self.docs:
            for token in set(doc):
                df[token] += 1

        n = len(self.docs)
        self.idf = {
            token: math.log(1.0 + (n - freq + 0.5) / (freq + 0.5))
            for token, freq in df.items()
        }

    def score(self, query: str) -> np.ndarray:
        q_tokens = tokenize_bm25(query)
        scores = np.zeros(len(self.docs), dtype=float)

        for i, tf in enumerate(self.tf):
            dl = self.lengths[i]
            for token in q_tokens:
                freq = tf.get(token, 0)
                if freq == 0:
                    continue
                idf = self.idf.get(token, 0.0)
                denom = freq + self.k1 * (
                    1.0 - self.b + self.b * dl / max(self.avgdl, 1e-9)
                )
                scores[i] += idf * (freq * (self.k1 + 1.0)) / denom

        return scores


bm25 = SimpleBM25([chunk["text"] for chunk in chunks])
print("BM25 indexado:", len(chunks), "chunks")


In [ ]:
class DenseRetriever:
    def __init__(self, chunks: list[dict[str, Any]], real: bool):
        self.chunks = chunks
        self.real = real

        if real:
            from sentence_transformers import SentenceTransformer
            import faiss

            self.model = SentenceTransformer(DENSE_MODEL_ID)
            vectors = self.model.encode(
                [f'passage: {c["text"]}' for c in chunks],
                normalize_embeddings=True,
                convert_to_numpy=True,
                show_progress_bar=False,
            ).astype("float32")
            self.index = faiss.IndexFlatIP(vectors.shape[1])
            self.index.add(vectors)
        else:
            self.vectorizer = TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=1,
            )
            x = self.vectorizer.fit_transform([c["text"] for c in chunks])

            max_components = min(x.shape[0] - 1, x.shape[1] - 1, 64)
            if max_components >= 2:
                self.svd = TruncatedSVD(
                    n_components=max_components,
                    random_state=SEED,
                )
                dense = self.svd.fit_transform(x)
                self.matrix = normalize(dense)
            else:
                self.svd = None
                self.matrix = normalize(x).toarray()

    def rank(self, query: str, depth: int) -> list[dict[str, Any]]:
        if self.real:
            q = self.model.encode(
                [f"query: {query}"],
                normalize_embeddings=True,
                convert_to_numpy=True,
                show_progress_bar=False,
            ).astype("float32")
            scores, indices = self.index.search(q, min(depth, len(self.chunks)))
            pairs = zip(indices[0], scores[0])
        else:
            q = self.vectorizer.transform([query])
            if self.svd is not None:
                qv = normalize(self.svd.transform(q))
            else:
                qv = normalize(q).toarray()
            scores = np.asarray(self.matrix @ qv[0]).reshape(-1)
            indices = np.argsort(-scores)[:depth]
            pairs = [(int(i), float(scores[i])) for i in indices]

        return [
            {**self.chunks[int(i)], "score": float(score)}
            for i, score in pairs
        ]


dense_retriever = DenseRetriever(chunks, RUN_REAL_RETRIEVAL)
print("Dense mode:", "E5 real" if RUN_REAL_RETRIEVAL else "TF-IDF+SVD offline")


In [ ]:
def bm25_rank(query: str, depth: int) -> list[dict[str, Any]]:
    scores = bm25.score(query)
    indices = np.argsort(-scores)[:depth]
    return [
        {**chunks[int(i)], "score": float(scores[i])}
        for i in indices
    ]


def reciprocal_rank_fusion(
    rankings: list[list[dict[str, Any]]],
    depth: int = CANDIDATE_DEPTH,
    constant: int = RRF_CONSTANT,
) -> list[dict[str, Any]]:
    fused = defaultdict(float)

    for ranking in rankings:
        for rank, item in enumerate(ranking, start=1):
            fused[item["chunk_id"]] += 1.0 / (constant + rank)

    ordered = sorted(
        fused.items(),
        key=lambda kv: (-kv[1], kv[0]),
    )[:depth]

    return [
        {**chunk_by_id[chunk_id], "score": float(score)}
        for chunk_id, score in ordered
    ]


def lexical_pair_score(query: str, text: str) -> float:
    q = set(tokenize_bm25(query))
    d = set(tokenize_bm25(text))
    if not q or not d:
        return 0.0
    return len(q & d) / len(q | d)


class Reranker:
    def __init__(self, real: bool):
        self.real = real
        if real:
            from sentence_transformers import CrossEncoder
            self.model = CrossEncoder(RERANKER_MODEL_ID)

    def rerank(
        self,
        query: str,
        candidates: list[dict[str, Any]],
        depth: int,
    ) -> list[dict[str, Any]]:
        if self.real:
            scores = self.model.predict([
                [query, item["text"]]
                for item in candidates
            ])
        else:
            scores = [
                lexical_pair_score(query, item["text"])
                for item in candidates
            ]

        rows = [
            {**item, "score": float(score)}
            for item, score in zip(candidates, scores)
        ]
        rows.sort(key=lambda x: (-x["score"], x["chunk_id"]))
        return rows[:depth]


reranker = Reranker(RUN_REAL_RERANKER)
print("Reranker mode:", "Cross-Encoder real" if RUN_REAL_RERANKER else "overlap léxico offline")


#### **5. Ejecutar A/B/C sobre las mismas 24 consultas**

Cada condición produce un ranking con `candidate_depth = 10`.

```text
A dense
B hybrid RRF
C hybrid RRF + reranker
```

La latencia se registra de dos maneras:

```text
total
-> costo end-to-end de la condición

por etapa
-> dense_ms
-> bm25_ms
-> rrf_ms
-> reranker_ms
```

Esto permite distinguir:

```text
"la condición C tarda más"
```

de una atribución más útil:

```text
"el costo incremental dominante proviene del reranker"
```

Las mediciones son instrumentación del experimento docente, no un microbenchmark de producción. Para benchmarking de rendimiento riguroso se necesitarían warm-up, múltiples repeticiones, control de hardware y sincronización explícita cuando corresponda.


In [ ]:
def elapsed_ms(start: float) -> float:
    return (time.perf_counter() - start) * 1000.0


def run_condition(
    condition: str,
    query: str,
) -> tuple[list[dict[str, Any]], dict[str, float]]:
    timings = {
        "dense_ms": 0.0,
        "bm25_ms": 0.0,
        "rrf_ms": 0.0,
        "reranker_ms": 0.0,
        "total_ms": 0.0,
    }

    total_start = time.perf_counter()

    start = time.perf_counter()
    dense = dense_retriever.rank(query, CANDIDATE_DEPTH)
    timings["dense_ms"] = elapsed_ms(start)

    if condition == "A_dense":
        ranking = dense

    elif condition == "B_hybrid_rrf":
        start = time.perf_counter()
        lexical = bm25_rank(query, CANDIDATE_DEPTH)
        timings["bm25_ms"] = elapsed_ms(start)

        start = time.perf_counter()
        ranking = reciprocal_rank_fusion(
            [dense, lexical],
            depth=CANDIDATE_DEPTH,
        )
        timings["rrf_ms"] = elapsed_ms(start)

    elif condition == "C_hybrid_rrf_reranker":
        start = time.perf_counter()
        lexical = bm25_rank(query, CANDIDATE_DEPTH)
        timings["bm25_ms"] = elapsed_ms(start)

        start = time.perf_counter()
        hybrid = reciprocal_rank_fusion(
            [dense, lexical],
            depth=CANDIDATE_DEPTH,
        )
        timings["rrf_ms"] = elapsed_ms(start)

        start = time.perf_counter()
        ranking = reranker.rerank(
            query,
            hybrid,
            depth=CANDIDATE_DEPTH,
        )
        timings["reranker_ms"] = elapsed_ms(start)

    else:
        raise ValueError(f"Condición desconocida: {condition}")

    timings["total_ms"] = elapsed_ms(total_start)
    return ranking, timings


CONDITIONS = [
    "A_dense",
    "B_hybrid_rrf",
    "C_hybrid_rrf_reranker",
]

retrieval_runs: dict[tuple[str, str], list[dict[str, Any]]] = {}
rows = []

for condition in CONDITIONS:
    for query_row in queries:
        qid = query_row["query_id"]
        ranking, timings = run_condition(
            condition,
            query_row["query"],
        )
        retrieval_runs[(condition, qid)] = ranking

        passage_ranking = expand_to_passage_ranking(ranking)
        relevant = qrels[qid]

        row = {
            "condition": condition,
            "query_id": qid,
            "difficulty": query_row.get("difficulty", "unknown"),
            **timings,
            "mrr": reciprocal_rank(passage_ranking, relevant),
            "ndcg@5": ndcg_at_k(passage_ranking, relevant, 5),
        }

        for k in TOP_K_VALUES:
            row[f"recall@{k}"] = recall_at_k(
                passage_ranking,
                relevant,
                k,
            )

        rows.append(row)

per_query_df = pd.DataFrame(rows)
per_query_df.head()


In [ ]:
def p95(series: pd.Series) -> float:
    return float(np.percentile(series.to_numpy(dtype=float), 95))


summary_df = (
    per_query_df
    .groupby("condition", as_index=False)
    .agg({
        "recall@1": "mean",
        "recall@3": "mean",
        "recall@5": "mean",
        "mrr": "mean",
        "ndcg@5": "mean",
        "total_ms": ["mean", p95],
    })
)

summary_df.columns = [
    "condition",
    "recall@1",
    "recall@3",
    "recall@5",
    "mrr",
    "ndcg@5",
    "latency_total_mean_ms",
    "latency_total_p95_ms",
]

latency_stage_df = (
    per_query_df
    .groupby("condition", as_index=False)
    .agg({
        "dense_ms": "mean",
        "bm25_ms": "mean",
        "rrf_ms": "mean",
        "reranker_ms": "mean",
        "total_ms": ["mean", p95],
    })
)

latency_stage_df.columns = [
    "condition",
    "dense_mean_ms",
    "bm25_mean_ms",
    "rrf_mean_ms",
    "reranker_mean_ms",
    "total_mean_ms",
    "total_p95_ms",
]

print("Tabla principal:")
display(summary_df)

print("\nDesglose de latencia por etapa:")
display(latency_stage_df)


In [ ]:
# Comparación pareada por consulta: las mismas queries atraviesan A, B y C.
pivot = per_query_df.pivot(
    index="query_id",
    columns="condition",
    values=["recall@3", "mrr", "ndcg@5"],
)

delta_rows = []
for qid in sorted(query_by_id):
    delta_rows.append({
        "query_id": qid,
        "difficulty": query_by_id[qid].get("difficulty"),
        "delta_recall3_B_vs_A":
            float(pivot.loc[qid, ("recall@3", "B_hybrid_rrf")]
                  - pivot.loc[qid, ("recall@3", "A_dense")]),
        "delta_recall3_C_vs_B":
            float(pivot.loc[qid, ("recall@3", "C_hybrid_rrf_reranker")]
                  - pivot.loc[qid, ("recall@3", "B_hybrid_rrf")]),
        "delta_mrr_B_vs_A":
            float(pivot.loc[qid, ("mrr", "B_hybrid_rrf")]
                  - pivot.loc[qid, ("mrr", "A_dense")]),
        "delta_mrr_C_vs_B":
            float(pivot.loc[qid, ("mrr", "C_hybrid_rrf_reranker")]
                  - pivot.loc[qid, ("mrr", "B_hybrid_rrf")]),
        "delta_ndcg5_B_vs_A":
            float(pivot.loc[qid, ("ndcg@5", "B_hybrid_rrf")]
                  - pivot.loc[qid, ("ndcg@5", "A_dense")]),
        "delta_ndcg5_C_vs_B":
            float(pivot.loc[qid, ("ndcg@5", "C_hybrid_rrf_reranker")]
                  - pivot.loc[qid, ("ndcg@5", "B_hybrid_rrf")]),
    })

delta_df = pd.DataFrame(delta_rows)
delta_df.sort_values("delta_ndcg5_C_vs_B").head(8)


#### **5.1 Incertidumbre: bootstrap pareado sobre diferencias por consulta**

Reportar únicamente:

```text
MRR(B) = ...
MRR(C) = ...
```

no informa cuán estable es la diferencia observada.

Como las mismas 24 consultas se evalúan bajo todas las condiciones, el objeto natural es el vector pareado:

$$
d_i = metric_i(C)-metric_i(B)
$$

El bootstrap remuestrea **consultas completas con reemplazo** y conserva el emparejamiento A/B/C.

Para cada comparación se reporta:

```text
delta_mean
CI95_low
CI95_high
```

Interpretación cuidadosa:

```text
IC que cruza 0
-> los datos son compatibles con mejoras y empeoramientos bajo este remuestreo

IC que no cruza 0
-> la dirección es más estable dentro de este benchmark

en ambos casos
-> 24 consultas siguen siendo evidencia limitada
-> el IC no demuestra generalización a otros corpus
```

No se presenta el bootstrap como una prueba mágica de "significancia". Es una cuantificación de incertidumbre condicionada al benchmark observado.


In [ ]:
def paired_bootstrap_delta(
    frame: pd.DataFrame,
    metric: str,
    condition_a: str,
    condition_b: str,
    n_resamples: int = BOOTSTRAP_RESAMPLES,
    alpha: float = BOOTSTRAP_ALPHA,
    seed: int = SEED,
) -> dict[str, Any]:
    a = (
        frame.loc[frame["condition"] == condition_a, ["query_id", metric]]
        .set_index("query_id")
        .sort_index()
    )
    b = (
        frame.loc[frame["condition"] == condition_b, ["query_id", metric]]
        .set_index("query_id")
        .sort_index()
    )

    if list(a.index) != list(b.index):
        raise ValueError("Las comparaciones pareadas requieren las mismas query_id.")

    deltas = (
        b[metric].to_numpy(dtype=float)
        - a[metric].to_numpy(dtype=float)
    )

    rng = np.random.default_rng(seed)
    n = len(deltas)
    bootstrap_means = np.empty(n_resamples, dtype=float)

    for i in range(n_resamples):
        sample_idx = rng.integers(0, n, size=n)
        bootstrap_means[i] = deltas[sample_idx].mean()

    low = float(np.quantile(bootstrap_means, alpha / 2))
    high = float(np.quantile(bootstrap_means, 1 - alpha / 2))

    return {
        "metric": metric,
        "comparison": f"{condition_b} - {condition_a}",
        "n_queries": n,
        "delta_mean": float(deltas.mean()),
        "ci95_low": low,
        "ci95_high": high,
        "ci_crosses_zero": bool(low <= 0.0 <= high),
        "n_resamples": n_resamples,
    }


bootstrap_rows = []
for metric in ["recall@3", "mrr", "ndcg@5"]:
    bootstrap_rows.append(
        paired_bootstrap_delta(
            per_query_df, metric,
            "A_dense", "B_hybrid_rrf",
        )
    )
    bootstrap_rows.append(
        paired_bootstrap_delta(
            per_query_df, metric,
            "B_hybrid_rrf", "C_hybrid_rrf_reranker",
        )
    )

bootstrap_ci_df = pd.DataFrame(bootstrap_rows)
bootstrap_ci_df


#### **Actividad 2 - No leas la tabla como una competencia**

Utiliza conjuntamente:

```text
medias agregadas + deltas por query + IC 95% bootstrap de la diferencia pareada + latencia por etapa
```

Para cada comparación responde:

```text
A -> B
¿Qué cambió realmente?
¿Podemos atribuir la diferencia a un solo componente?
¿El IC de la diferencia cruza 0?

B -> C
¿Qué componente nuevo aparece?
¿Cambió cobertura, orden o ambos?
¿La dirección del cambio es estable bajo bootstrap?

C
¿La mejora, si existe, compensa la latencia?
¿Qué etapa explica el costo incremental?
```

No utilices únicamente:

```text
"el número más alto gana"
```

Tampoco utilices:

```text
"el IC no cruza 0, entonces el sistema es universalmente mejor"
```

El alcance sigue limitado a este benchmark docente.


#### **6. Taxonomía de fallos para retrieval**

No todo fallo significa lo mismo.

```text
RETRIEVAL_MISS
-> ninguna evidencia relevante aparece dentro de candidate_depth

RANKING_ERROR
-> existe evidencia relevante entre candidatos,
   pero no llega al top-3

PARTIAL_RECALL
-> top-3 contiene solo parte de la evidencia relevante

RETRIEVAL_OK
-> top-3 contiene toda la evidencia relevante
```


In [ ]:
def classify_retrieval_failure(
    ranked_chunks: list[dict[str, Any]],
    relevant_passages: list[str],
    top_k: int = 3,
) -> str:
    passage_ranking = expand_to_passage_ranking(ranked_chunks)

    recall_candidate = recall_at_k(
        passage_ranking,
        relevant_passages,
        min(CANDIDATE_DEPTH, len(passage_ranking)),
    )
    recall_top = recall_at_k(
        passage_ranking,
        relevant_passages,
        top_k,
    )

    if recall_candidate == 0:
        return "RETRIEVAL_MISS"
    if recall_top == 0:
        return "RANKING_ERROR"
    if recall_top < 1:
        return "PARTIAL_RECALL"
    return "RETRIEVAL_OK"


failure_rows = []
for condition in CONDITIONS:
    for qid in sorted(query_by_id):
        failure_rows.append({
            "condition": condition,
            "query_id": qid,
            "failure_type": classify_retrieval_failure(
                retrieval_runs[(condition, qid)],
                qrels[qid],
            ),
        })

failure_df = pd.DataFrame(failure_rows)

pd.crosstab(
    failure_df["condition"],
    failure_df["failure_type"],
)


In [ ]:
# Casos que conviene inspeccionar durante la discusión.
analysis_df = (
    per_query_df
    .merge(failure_df, on=["condition", "query_id"])
    .sort_values(
        ["condition", "recall@3", "mrr", "ndcg@5"],
        ascending=[True, True, True, True],
    )
)

analysis_df.head(12)


#### **7. Evaluación de generation: corrección, citas y faithfulness**

Una evaluación end-to-end no debe confundir tres preguntas:

```text
correctness
-> ¿la respuesta contiene la información necesaria?

citation correctness
-> ¿las citas apuntan a evidencia relevante?

faithfulness
-> ¿los claims están respaldados por el contexto realmente entregado?
```

El cuaderno no presenta un overlap léxico como si fuera una métrica semántica perfecta.

Se incluye un `reference_token_f1_proxy` únicamente para depurar y comparar salidas de forma reproducible.


In [ ]:
def normalized_tokens(text: str) -> list[str]:
    return [m.group(0).lower() for m in TOKEN_RE.finditer(text)]


def reference_token_f1_proxy(
    prediction: str,
    reference: str,
) -> float:
    pred = Counter(normalized_tokens(prediction))
    gold = Counter(normalized_tokens(reference))

    overlap = sum((pred & gold).values())
    pred_total = sum(pred.values())
    gold_total = sum(gold.values())

    if pred_total == 0 or gold_total == 0:
        return 0.0

    precision = overlap / pred_total
    recall = overlap / gold_total

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def citation_metrics(
    citations: list[str],
    context_chunks: list[dict[str, Any]],
    relevant_passages: list[str],
) -> dict[str, float]:
    context_ids = {c["chunk_id"] for c in context_chunks}
    relevant = set(relevant_passages)

    if not citations:
        return {
            "citation_in_context_rate": 0.0,
            "citation_support_rate": 0.0,
        }

    in_context = 0
    supported = 0

    for citation in citations:
        if citation in context_ids:
            in_context += 1
            chunk = chunk_by_id[citation]
            if relevant & set(chunk["passage_ids"]):
                supported += 1

    n = len(citations)
    return {
        "citation_in_context_rate": in_context / n,
        "citation_support_rate": supported / n,
    }


assert math.isclose(
    reference_token_f1_proxy("dns institucional", "dns institucional"),
    1.0,
)
print("Proxies deterministas: OK")


#### **8. Generación real opcional**

El núcleo de Semana 7 puede ejecutarse sin LLM.

Cuando se activa:

```bash
CC0F4_RUN_REAL_LLM=1
```

se reutiliza el generador pequeño de Semana 5 sobre seis consultas de auditoría.

La salida solicitada es:

```json
{
  "answer": "...",
  "citations": ["D01-C01"]
}
```

Las citas deben referirse únicamente a chunks entregados en el contexto.

##### **Por qué `RAG_TOP_K = 3` en el experimento canónico**

`RAG_TOP_K=3` no se presenta como un valor óptimo universal. Se fija porque:

```text
Recall@3
-> es una métrica de cobertura central del experimento

RAG_TOP_K=3
-> hace que la ventana observada por el generador corresponda al mismo corte

top-k fijo en A/B/C
-> evita cambiar simultáneamente retrieval y presupuesto de contexto
```

Después del experimento canónico, el **Ejercicio 4B** estudia sensibilidad con:

```text
RAG_CONTEXT_TOP_K_VALUES = [1, 3, 5]
```

En esa extensión debe mantenerse fijo todo lo demás y volver a medir tanto generación como costo.

Esta extensión no implementa un agente, reparación iterativa ni LLM-as-Judge.


In [ ]:
def build_context(
    ranked_chunks: list[dict[str, Any]],
    top_k: int = RAG_TOP_K,
) -> str:
    blocks = []
    for chunk in ranked_chunks[:top_k]:
        blocks.append(
            f'[{chunk["chunk_id"]}] '
            f'passages={",".join(chunk["passage_ids"])}\n'
            f'{chunk["text"]}'
        )
    return "\n\n".join(blocks)


def parse_json_object(text: str) -> dict[str, Any] | None:
    text = text.strip()
    try:
        value = json.loads(text)
        return value if isinstance(value, dict) else None
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not match:
            return None
        try:
            value = json.loads(match.group(0))
            return value if isinstance(value, dict) else None
        except json.JSONDecodeError:
            return None


def generate_real_outputs() -> list[dict[str, Any]]:
    if not RUN_REAL_LLM:
        return []

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        GENERATOR_MODEL_ID,
        torch_dtype="auto",
        device_map="auto",
    )

    rows = []

    system = (
        "Responde usando únicamente el contexto proporcionado. "
        "Devuelve SOLO JSON con campos answer y citations. "
        "citations debe contener únicamente chunk_id presentes en el contexto. "
        "Si la evidencia es insuficiente, indícalo explícitamente."
    )

    for condition in CONDITIONS:
        for qid in AUDIT_QUERY_IDS:
            query = query_by_id[qid]["query"]
            ranking = retrieval_runs[(condition, qid)]
            context = build_context(ranking)

            user = (
                f"Pregunta:\n{query}\n\n"
                f"Contexto:\n{context}\n\n"
                "Formato:\n"
                '{"answer":"...", "citations":["..."]}'
            )

            messages = [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ]

            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
            ).to(model.device)

            start = time.perf_counter()
            with torch.no_grad():
                output = model.generate(
                    **inputs,
                    max_new_tokens=160,
                    do_sample=False,
                )
            latency_ms = (time.perf_counter() - start) * 1000.0

            generated = output[0, inputs["input_ids"].shape[1]:]
            raw = tokenizer.decode(generated, skip_special_tokens=True)
            parsed = parse_json_object(raw) or {}

            answer = str(parsed.get("answer", ""))
            citations = parsed.get("citations", [])
            if not isinstance(citations, list):
                citations = []

            ref = reference_by_id[qid]
            citation_eval = citation_metrics(
                [str(x) for x in citations],
                ranking[:RAG_TOP_K],
                qrels[qid],
            )

            rows.append({
                "condition": condition,
                "query_id": qid,
                "answer": answer,
                "citations": json.dumps(citations, ensure_ascii=False),
                "raw_output": raw,
                "reference_token_f1_proxy":
                    reference_token_f1_proxy(
                        answer,
                        ref["reference_answer"],
                    ),
                **citation_eval,
                "generation_latency_ms": latency_ms,
            })

    return rows


generation_rows = generate_real_outputs()
generation_df = pd.DataFrame(generation_rows)

if generation_df.empty:
    print(
        "Generación real desactivada. "
        "Se crearán plantillas de auditoría sin afirmar resultados del LLM."
    )
else:
    generation_df.head()


#### **9. Auditoría manual de faithfulness**

La auditoría manual fuerza a observar el nivel que un score automático intenta aproximar.

Para cada salida seleccionada:

```text
respuesta
-> separar claims atómicos
-> localizar evidencia
-> supported_by_context = 0/1
-> registrar supporting_chunk_id
-> anotar motivo
```

##### **Criterio operativo de aceptación**

Marca:

```text
supported_by_context = 1
```

solo cuando **al menos un chunk del contexto entregado contiene evidencia explícita suficiente para respaldar el claim sin recurrir a conocimiento externo ni a una inferencia no justificada**.

Marca:

```text
supported_by_context = 0
```

cuando la evidencia está ausente, contradice el claim o solo respalda una parte material del claim.

Si una oración contiene una parte respaldada y otra no respaldada:

```text
NO usar 0.5
-> dividirla en claims atómicos
-> evaluar cada claim por separado
```

`supporting_chunk_id` debe identificar el chunk concreto utilizado para justificar el `1`.

Este criterio no elimina toda subjetividad, pero hace la auditoría reproducible y permite discutir desacuerdos entre evaluadores.


In [ ]:
audit_rows = []

if generation_df.empty:
    for condition in CONDITIONS:
        for qid in AUDIT_QUERY_IDS:
            audit_rows.append({
                "condition": condition,
                "query_id": qid,
                "response": "",
                "claim": "",
                "supported_by_context": "",
                "supporting_chunk_id": "",
                "failure_type": "",
                "notes": "",
            })
else:
    for row in generation_rows:
        audit_rows.append({
            "condition": row["condition"],
            "query_id": row["query_id"],
            "response": row["answer"],
            "claim": "",
            "supported_by_context": "",
            "supporting_chunk_id": "",
            "failure_type": "",
            "notes": "",
        })

audit_df = pd.DataFrame(audit_rows)
audit_df.head()


#### **Actividad 3 - Clasifica un fallo**

Para una consulta dada, decide cuál explicación es defendible:

```text
RETRIEVAL_MISS
RANKING_ERROR
PARTIAL_RECALL
GENERATION_IGNORE
UNSUPPORTED_CLAIM
CITATION_ERROR
INCOMPLETE_ANSWER
```

No asignes una categoría si no tienes la evidencia necesaria para distinguirla.


#### **10. Exportar evidencia reproducible**

La corrida registra hashes de los datos, modos real/offline y métricas.

```text
resultado + configuración + hashes + limitaciones
```

es más útil que guardar únicamente una tabla copiada a una diapositiva.


In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(65536), b""):
            h.update(block)
    return h.hexdigest()


per_query_path = RESULTS_DIR / "retrieval_per_query.csv"
summary_path = RESULTS_DIR / "retrieval_summary.csv"
audit_path = RESULTS_DIR / "faithfulness_audit_template.csv"
bootstrap_path = RESULTS_DIR / "bootstrap_pairwise_ci.csv"
latency_path = RESULTS_DIR / "latency_by_stage.csv"

per_query_df.to_csv(per_query_path, index=False)
summary_df.to_csv(summary_path, index=False)
audit_df.to_csv(audit_path, index=False)
bootstrap_ci_df.to_csv(bootstrap_path, index=False)
latency_stage_df.to_csv(latency_path, index=False)

generation_path = RESULTS_DIR / "generation_outputs.csv"
if not generation_df.empty:
    generation_df.to_csv(generation_path, index=False)

manifest = {
    "week": 7,
    "seed": SEED,
    "config": {
        "target_words": TARGET_WORDS,
        "overlap_passages": OVERLAP_PASSAGES,
        "candidate_depth": CANDIDATE_DEPTH,
        "top_k_values": TOP_K_VALUES,
        "rrf_constant": RRF_CONSTANT,
        "rag_top_k": RAG_TOP_K,
        "rag_context_top_k_values": RAG_CONTEXT_TOP_K_VALUES,
        "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
        "bootstrap_alpha": BOOTSTRAP_ALPHA,
        "dense_model_id": DENSE_MODEL_ID,
        "reranker_model_id": RERANKER_MODEL_ID,
        "generator_model_id": GENERATOR_MODEL_ID,
        "run_real_retrieval": RUN_REAL_RETRIEVAL,
        "run_real_reranker": RUN_REAL_RERANKER,
        "run_real_llm": RUN_REAL_LLM,
    },
    "hashes": {
        "corpus_semana4.jsonl":
            sha256_file(DATA4_DIR / "corpus_semana4.jsonl"),
        "queries_semana4.jsonl":
            sha256_file(DATA4_DIR / "queries_semana4.jsonl"),
        "qrels_semana4.json":
            sha256_file(DATA4_DIR / "qrels_semana4.json"),
        "reference_answers_semana7.jsonl":
            sha256_file(DATA7_DIR / "reference_answers_semana7.jsonl"),
    },
    "retrieval_summary": summary_df.to_dict(orient="records"),
    "latency_by_stage": latency_stage_df.to_dict(orient="records"),
    "bootstrap_pairwise_ci": bootstrap_ci_df.to_dict(orient="records"),
    "generation_executed": not generation_df.empty,
    "limitations": [
        "Benchmark docente de 24 consultas.",
        "Los qrels son binarios.",
        "El modo offline valida software, no representa E5/Cross-Encoder.",
        "reference_token_f1_proxy no es una métrica semántica universal.",
        "Faithfulness requiere auditoría de claims o un evaluador adicional validado.",
    ],
}

manifest_path = RESULTS_DIR / "latest_run.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Generado:", summary_path)
print("Generado:", per_query_path)
print("Generado:", audit_path)
print("Generado:", bootstrap_path)
print("Generado:", latency_path)
if not generation_df.empty:
    print("Generado:", generation_path)
print("Generado:", manifest_path)


#### **11. Interpretación obligatoria**

Completa después de ejecutar:

```text
1. ¿Qué condición obtuvo mayor Recall@3?
2. ¿Qué condición obtuvo mayor MRR?
3. ¿Qué condición obtuvo mayor nDCG@5?
4. ¿Las tres métricas ordenan los sistemas igual?
5. ¿Qué consultas mejoraron con RRF?
6. ¿Qué consultas empeoraron?
7. ¿El reranker recuperó evidencia nueva o principalmente reordenó candidatos?
8. ¿El IC 95% bootstrap de B->C cruza 0 para MRR o nDCG@5?
9. ¿Qué etapa explica la mayor parte de la latencia incremental de C?
10. ¿Qué ocurrió con latency total p95?
11. ¿Qué failure mode aparece con mayor frecuencia?
12. ¿Qué conclusión NO puedes defender con 24 consultas aunque un IC no cruce 0?.
```


#### **12. Ejercicios de práctica autónoma**

##### **Ejercicio 1 - Precision@k**

Implementa `precision_at_k()` y explica por qué no se usa como métrica principal del experimento canónico.

##### **Ejercicio 2 - nDCG con relevancia graduada**

Construye un ejemplo sintético con grados:

```text
0 = irrelevante
1 = parcialmente relevante
2 = muy relevante
```

No modifiques los qrels reales de Semana 7.

##### **Ejercicio 3 - Consultas clear vs ambiguous**

Compara las métricas agregadas por `difficulty`.

Pregunta:

> ¿la dificultad declarada en el benchmark está asociada con diferencias observables en retrieval?.

No conviertas una diferencia pequeña en conclusión causal.

##### **Ejercicio 4A - Trade-off de retrieval top-k**

Evalúa el ranking en:

```text
1
3
5
10
```

y discute cobertura frente a profundidad de recuperación.

Aquí todavía no estás cambiando necesariamente cuánto contexto consume el generador.

##### **Ejercicio 4B - Sensibilidad del contexto de generación**

Si ejecutas el LLM real, repite las seis consultas de auditoría con:

```text
RAG_CONTEXT_TOP_K_VALUES = [1, 3, 5]
```

Para cada valor construye el contexto con:

```python
build_context(ranking, top_k=k)
```

y vuelve a medir:

```text
reference_token_f1_proxy
citation_in_context_rate
citation_support_rate
generation_latency_ms
auditoría manual de claims
```

Pregunta:

> ¿más contexto mejora coverage o introduce ruido que el generador no utiliza correctamente?.

No mezcles este experimento con la comparación canónica A/B/C: primero se mantiene `RAG_TOP_K=3` fijo para atribuir cambios al retriever/reranker.

##### **Ejercicio 5 - Auditoría interevaluador**

Selecciona tres respuestas y pide a dos personas que auditen los claims de forma independiente usando el criterio de la sección 9.

Discute:

```text
acuerdo
desacuerdo
definición de claim
criterio de soporte
```

Este ejercicio prepara la discusión sobre métricas automáticas y human judgment.


#### **13. Referencias para E3**

```text
Manning, Raghavan, Schütze
Introduction to Information Retrieval
Chapter 8

RAGAS
arXiv:2309.15217

ARES
arXiv:2311.09476

RAGChecker
arXiv:2408.08067
```

No se instalan estos frameworks como requisito del lunes.

Primero:

```text
definición
-> implementación pequeña
-> test
-> experimento
-> limitación
```

Después:

```text
framework
-> automatiza parte del proceso
```


#### **14. Cierre**

Al terminar Semana 7 debes poder defender:

```text
retrieval quality != generation quality

Recall@k != MRR != nDCG

mejor ranking != mejor respuesta automáticamente

correctness != faithfulness

citation in context != citation supports claim

aggregate metric != error analysis

software validation != experimental evidence

LLM-as-Judge != ground truth
```

La pregunta que debe quedar antes del proyecto parcial es:

> ¿Qué evidencia tengo para sostener que mi sistema funciona y qué parte de esa afirmación todavía no he medido?.
